# Import

In [ ]:
import colorsys
from itertools import combinations, product, permutations
import json
import matplotlib
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from matplotlib.ticker import FuncFormatter
from matplotlib import (pylab, rc, transforms)
import numpy as np
import openturns as ot
import os
import pandas as pd
from pprint import pprint
import random
import requests
import scipy
from scipy import (stats, linalg, special, integrate)
from scipy.integrate import (trapezoid, cumulative_trapezoid)
from scipy.interpolate import interp1d
from scipy.optimize import minimize
from scipy.signal import argrelextrema
from scipy.stats import (anderson, energy_distance, gaussian_kde, kendalltau, ks_2samp, 
                         lognorm, norm, pearsonr, rv_continuous, spearmanr, wasserstein_distance)
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, r2_score
from sklearn.preprocessing import LabelEncoder
import string
import time
from tqdm import tqdm
from typing import Optional
import warnings
from xgboost import XGBRegressor, XGBClassifier, plot_importance

matplotlib.rcParams['figure.dpi'] = 1200
alphabet = string.ascii_lowercase
generate_dontread = False

In [ ]:
import sys
sys.path.insert(0, '../src')
sys.path.insert(0, '../../shared')

from funcs_unit_conversion import hi2, hi3, lo2

from customstats import weighted_lognorm_fit, shapiro_wilk_weighted, _royston_pvalue, empirical_metadata, NestedDictValues, weighted_ecdf, estimate_maxima, weighted_kurtosis, weighted_skew, wasserstein1_weighted, wasserstein2_weighted, weighted_mean, weighted_var, weighted_distance_norm, weighted_quantile, weighted_bw, weighted_std

from datageneration import (random_samples, generate_random_numbers, random_irregular_dataset, random_logcount)

from datavisualization import scale_lightness

from funcs_unit_conversion import dict_unitconv, dict_unittype, consistent_units, str2valunit


In [ ]:
rankth = {}

for i in np.arange(1,100):
    if i % 10 == 1:
        rankth[i] = f'{i}st'
    elif i % 10 == 2:
        rankth[i] = f'{i}nd'
    elif i % 10 == 3:
        rankth[i] = f'{i}rd'
    else:
        rankth[i] = f'{i}th'
rankth[11] = '11th'
rankth[12] = '12th'
rankth[13] = '13th'



In [ ]:
# read in dictionary with labels for each metric
with open('../src/dct_metriclabels.json', 'r') as file:
    dct_metriclabels = json.load(file)


In [ ]:
PE = ['Normal', 'Lognormal', 'KDE']
WT = ['Uniform', 'Variable']
PEWT = []
for pe, wt in product(PE, WT):
    PEWT.append(f'{pe}, {wt}')

dct_colors = {pewt: color for pewt, color in zip(PEWT, sns.color_palette('Paired', len(PEWT)))}


# Visualize reduction strategies

In [ ]:
data = np.array([12.12475736,  5.21457612, 12.97857192, 12.59576432, 12.59383363,
        8.21634167, 12.69950401,  5.50794499,  6.11143734, 12.14935398,
       12.54681013, 13.31896526,  6.43022792,  5.29897256,  8.13255968,
       13.97769912,  6.18196489,  8.90866674, 10.51981915,  7.005553  ])
xplot = np.linspace(0,20,1_000)
q3 = np.quantile(data, 0.75)
func = gaussian_kde(data)
yplot = func.pdf(xplot)

fig, axes = plt.subplots(2,1)
# set max ECC cap
ax = axes[0]
ax.plot(xplot, yplot, color='gray', linestyle='--')
ind = np.argmin(np.abs(xplot-q3))
ax.fill_between(xplot[:ind], yplot[:ind], color='tab:blue', alpha=0.5)


# material reduction
ax = axes[1]
ax.plot(xplot, yplot, color='gray', linestyle='--')
ax.fill_between(xplot*0.75, yplot, color='tab:blue', alpha=0.5)


# format
for i in range(2):
    ax = axes[i]
    ax.spines[['top', 'right' ,'left']].set_visible(False)
    ax.get_yaxis().set_visible(False)
    ax.set_xticks([])
    ax.set_ylim(0,)

plt.savefig('../outputs/figures/CompareUQMethods_DEF_VisualizeReductionStrategies.png', bbox_inches='tight')


# Import data

In [ ]:
# read data from file
with open('../data/processed/DATA_all.json', 'r') as file:
    DATA_all = json.load(file)

# convert lists to arrays
for dataset in DATA_all:
    DATA_all[dataset]['data'] = np.array(DATA_all[dataset]['data'])
    DATA_all[dataset]['weights'] = np.array(DATA_all[dataset]['weights'])

# read lists that trim data
with open('../data/processed/datasets_outliers.json', 'r') as f:
    datasets_outliers = json.load(f)
with open('../data/processed/datasets_trimto10k.json', 'r') as f:
    datasets_trimto10k = json.load(f)

# create new, trimmed dataset
datasets = DATA_all.keys()
datasets_trimmed = [ele for ele in datasets if ele not in datasets_trimto10k and ele not in datasets_outliers]
DATA = {key: DATA_all[key] for key in datasets_trimmed}

In [ ]:
# put all metrics into a single dataframe
dataset = list(DATA.keys())[0]
df_metrics = pd.DataFrame(columns=DATA[dataset]['metrics'].keys())
for dataset in tqdm(DATA, total=len(DATA)):
    df_metrics.loc[dataset] = DATA[dataset]['metrics']

In [ ]:
# read in combos and make sure all datasets in combos are in DATA
nmats = 4
with open('../data/processed/combos.txt') as file:
    combos = str(file.readlines())
combos = [ele for ele in combos.split("'") if ele.startswith('dataset')]
combos = np.array(combos).reshape(int(len(combos)/nmats), nmats)

count = 0
for dataset in combos.flatten():
    if dataset not in DATA:
        count += 1
if count > 0:
    raise ValueError(f'{count} values from combos are not in DATA')

In [ ]:
# fit probabilistic models to each distribution (KDE, Normal, Lognormal) + (Uniform, Variable, Dirichlet)
# create ECC models - new version with three probability estimation methods (pe) and three weighting methods (wt)
# df_ecc = pd.DataFrame()
# nsamples = 10_000
prob_models = {k:{} for k in DATA.keys()}
PE = ['Normal', 'Lognormal', 'KDE']
WT = ['Uniform', 'Variable']

PEWT = []
for pe, wt in product(PE, WT):
    PEWT.append(f'{pe}, {wt}')

logfit_offset = 0.5


with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for dataset in tqdm(DATA, total=len(DATA)):
        #wt 1 - uniform weights
        #wt 2 - variable weights

        X = DATA[dataset]['data']
        W_uni = np.ones_like(X)/len(X)
        W_var = DATA[dataset]['weights']
        
        for W, wt in zip([W_uni, W_var], ['Uniform', 'Variable']):

            #Normal
            pe = 'Normal'
            pewt = f'{pe}, {wt}'
            mean = np.sum(X*W)/np.sum(W)
            std = weighted_std(X, W)
            func = norm(loc=mean, scale=std)
            prob_models[dataset][pewt] = func

            #Lognormal
            pe = 'Lognormal'
            pewt = f'{pe}, {wt}'
            shape, loc, scale = weighted_lognorm_fit(X+logfit_offset, weights=W, method='MLE')
            func = lognorm(s=shape, loc=loc-logfit_offset, scale=scale)
            prob_models[dataset][pewt] = func

            #KDE
            pe = 'KDE'
            pewt = f'{pe}, {wt}'
            bw = weighted_bw(X, W, bw_method='scott')
            func = gaussian_kde(X, bw_method=1.0, weights=W)
            bw_base = (func.covariance**0.5)[0][0]
            func.set_bandwidth(bw/bw_base)
            prob_models[dataset][pewt] = func

# pLCA

ECI metrics
- Mean ECI percentage
- Percent of runs where dataset is #1, #2, #3, #4 contributor to wbECI

Variance metrics
- Uncertainty index
- Uncertainty index rank
- Uncertainty index normalized to mean

EC reduction metrics
- Percent of runs where dataset is #1, #2, #3, #4 for expected EC reduction with material reduction
- Percent of runs where dataset is #1, #2, #3, #4 for expected EC reduction with ECC cap
- Mean EC reduction from material reduction
- Mean EC reduction from ECC cap

## run pLCA

In [ ]:
neccs = 1000
results_meta = {k:{} for k in PEWT}
capecc = 0.75
matred = 0.25

for nplca, datasets in tqdm(enumerate(combos), total=len(combos)):
    # Wdict = {k:v for (k, v) in zip(datasets, np.random.dirichlet(np.ones(nmats)))}
    Wdict = {k:1.0 for k in datasets}
    for pewt in PEWT:
        results_plca = {k:{} for k in datasets}
        ######################################################
        # add identifying information to results_plca
        ######################################################
        for dataset in datasets:
            results_plca[dataset]['plca'] = nplca
            results_plca[dataset]['pewt'] = pewt
            results_plca[dataset]['dataset'] = dataset
            # results_plca[dataset]['w'] = Wdict[dataset]
            # DATA[dataset]['metrics_plca'] = empirical_metadata(DATA[dataset]['data']*Wdict[dataset], DATA[dataset]['weights'])

        ######################################################
        # deterministic analysis - randomly selecting values from each dataset
        ######################################################
        # df_det = pd.DataFrame(columns=datasets)
        # dct_det = {}
        # for dataset in datasets:
        #     df_det[dataset] = np.random.choice(DATA[dataset]['data'], size=neccs)*Wdict[dataset]
        #     dct_det[dataset] = np.mean(DATA[dataset]['data'])*Wdict[dataset]

        # #percent of runs where dataset is #1, #2, #3, #4
        # df_detrank = pd.DataFrame(index=np.arange(nmats).astype(float)+1, columns=datasets)
        # for dataset in datasets:
        #     df_detrank[dataset] = df_det.rank(axis=1, ascending=False)[dataset].value_counts()
        # df_detrank = df_detrank.fillna(0)/neccs

        # for dataset in datasets:
        #     for i in df_detrank.index:
        #         results_plca[dataset][f'eci_detrank_{int(i)}'] = df_detrank.loc[i, dataset]
        #     results_plca[dataset]['eci_detrank'] = list(pd.Series(dct_det).sort_values(ascending=False).index).index(dataset)+1

        ######################################################
        # eci percentage mean, rank, normalized to mean
        ######################################################
        df_eci = pd.DataFrame()
        for dataset, w in zip(datasets, W):
            func = prob_models[dataset][pewt]
            if pewt.startswith('KDE'):
                eccs = func.resample(neccs)[0]
                while np.min(eccs) <= 0:
                    posvals = np.array([ele for ele in eccs if ele > 0])
                    nneg = len(eccs) - len(posvals)
                    eccs = np.concatenate([posvals, func.resample(nneg)[0]])
            else:
                eccs = func.rvs(neccs)
                while np.min(eccs) <= 0:
                    posvals = np.array([ele for ele in eccs if ele > 0])
                    nneg = len(eccs) - len(posvals)
                    eccs = np.concatenate([posvals, func.rvs(nneg)])
            df_eci[dataset] = eccs*Wdict[dataset]
        df_eci_perc = df_eci.copy()
        df_eci_perc = (df_eci_perc.T/df_eci.sum(axis=1)).T

        for dataset in datasets:
            results_plca[dataset]['eci_mean'] = df_eci.mean()[dataset]
            results_plca[dataset]['eci_std'] = df_eci.std()[dataset]
            results_plca[dataset]['eci_stdnorm'] = (df_eci.std()/df_eci.std().mean())[dataset]
            results_plca[dataset]['eci_stdrank'] = list(df_eci.std().sort_values(ascending=False).index).index(dataset)+1
            results_plca[dataset]['eci_cov'] = (df_eci.std()/df_eci.mean())[dataset]
            results_plca[dataset]['eci_meanrank'] = list(df_eci.mean().sort_values(ascending=False).index).index(dataset)+1
            results_plca[dataset]['eci_meannorm'] = (df_eci.mean()/df_eci.mean().mean())[dataset]

            results_plca[dataset]['eci_perc_mean'] = df_eci_perc.mean()[dataset]
            results_plca[dataset]['eci_perc_std'] = df_eci_perc.std()[dataset]
            results_plca[dataset]['eci_perc_rank'] = list(df_eci_perc.mean().sort_values(ascending=False).index).index(dataset)+1
            results_plca[dataset]['eci_perc_norm'] = (df_eci_perc.mean()/df_eci_perc.mean().mean())[dataset]

        results_plca[dataset]['wbeci_mean'] = np.mean(df_eci.sum(axis=1).values)
        results_plca[dataset]['wbeci_stdev'] = np.std(df_eci.sum(axis=1).values)
        
        #percent of runs where dataset is #1, #2, #3, #4
        df_rank = pd.DataFrame(index=np.arange(nmats).astype(float)+1, columns=datasets)
        for dataset in datasets:
            df_rank[dataset] = df_eci.rank(axis=1, ascending=False)[dataset].value_counts()
        df_rank = df_rank.fillna(0)/neccs

        for dataset in datasets:
            for i in df_rank.index:
                results_plca[dataset][f'eci_rank_{int(i)}'] = df_rank.loc[i, dataset]

        ######################################################
        #uncertainty index
        ######################################################
        UI = pd.Series(index=datasets)
        for dataset in datasets:
            eci_total = df_eci.sum(axis=1).values
            eci_run = (eci_total - df_eci[dataset] + df_eci[dataset].median()).values
            ui = 1 - np.var(eci_run)/np.var(eci_total)
            results_plca[dataset]['ui'] = ui
            UI[dataset] = ui

        for dataset in datasets:
            results_plca[dataset]['ui_rank'] = UI.rank(ascending=False)[dataset]
            results_plca[dataset]['ui_norm'] = (UI/np.mean(UI))[dataset]

        ######################################################
        # ECC cap
        ######################################################
        df_red = df_eci.copy()
        df_redperc = pd.DataFrame(index=df_eci.index, columns=datasets)
        for dataset in datasets:
            # replace values that need replacing
            func = prob_models[dataset][pewt]
            upper_limit = np.quantile(df_eci[dataset], capecc)
            inds = np.array([])
            while np.max(df_red[dataset]) >= upper_limit:
                ind = df_red[dataset].loc[df_red[dataset] >= upper_limit].index
                inds = np.concatenate([inds, ind])

                if pewt.startswith('KDE'):
                    new_eccs = func.resample(len(ind))[0]
                    while np.min(new_eccs) <= 0:
                        posvals = np.array([ele for ele in new_eccs if ele > 0])
                        nneg = len(new_eccs) - len(posvals)
                        new_eccs = np.concatenate([posvals, func.resample(nneg)[0]])
                else:
                    new_eccs = func.rvs(len(ind))
                    while np.min(new_eccs) <= 0:
                        posvals = np.array([ele for ele in new_eccs if ele > 0])
                        nneg = len(new_eccs) - len(posvals)
                        new_eccs = np.concatenate([posvals, func.rvs(nneg)])

                df_red.loc[ind, dataset] = new_eccs*Wdict[dataset]
            inds = list(set(inds))
            inds.sort()

            # get statistics for the cases in which reduction happens
            oldtotal = df_eci.loc[inds].sum(axis=1)
            newtotal = oldtotal - df_eci.loc[inds, dataset] + df_red.loc[inds, dataset]
            reduction = newtotal - oldtotal
            reduction_perc = reduction / oldtotal
            df_redperc.loc[inds, dataset] = reduction
            results_plca[dataset]['capecc_red_mean'] = np.mean(reduction)
            results_plca[dataset]['capecc_red_stdev'] = np.std(reduction)
            results_plca[dataset]['capecc_perc_mean'] = np.mean(reduction_perc)
            results_plca[dataset]['capecc_perc_stdev'] = np.std(reduction_perc)
            # results_plca[dataset]['capecc_redlikelihood'] = len(reduction[reduction<0])/len(reduction)

        # percent of runs where reduction strategy is most to least effective. Percentages look off because not all reduction strategies apply in all scenarios
        df_rank = pd.DataFrame(index=np.arange(nmats).astype(float)+1, columns=datasets)
        for dataset in datasets:
            df_rank[dataset] = df_redperc.rank(axis=1)[dataset].value_counts()
        df_rank = df_rank.fillna(0)/neccs/(1-capecc)
        for dataset in datasets:
            for i in df_rank.index:
                results_plca[dataset][f'capecc_rank_{int(i)}'] = df_rank.loc[i, dataset]
            results_plca[dataset]['capecc_rank'] = list(df_redperc.mean().sort_values(ascending=True).index).index(dataset)+1
            results_plca[dataset]['capecc_norm'] = (df_redperc.mean()/np.abs(df_redperc.mean().mean()))[dataset]

        ######################################################
        # Material reduction
        ######################################################
        df_red = df_eci*(1-matred)
        df_redperc = pd.DataFrame(index=df_eci.index, columns=datasets)
        oldtotal = df_eci.sum(axis=1)
        for dataset in datasets:
            newtotal = oldtotal - df_eci[dataset] + df_red[dataset]
            reduction = newtotal - oldtotal
            reduction_perc = reduction / oldtotal
            df_redperc[dataset] = reduction
            results_plca[dataset]['matred_red_mean'] = np.mean(reduction)
            results_plca[dataset]['matred_red_stdev'] = np.std(reduction)
            results_plca[dataset]['matred_perc_mean'] = np.mean(reduction_perc)
            results_plca[dataset]['matred_perc_stdev'] = np.std(reduction_perc)
            # results_plca[dataset]['matred_redlikelihood'] = len(reduction[reduction<0])/len(reduction)

        # percent of runs where reduction strategy is most to least effective. Percentages look off because not all reduction strategies apply in all scenarios
        df_rank = pd.DataFrame(index=np.arange(nmats).astype(float)+1, columns=datasets)
        for dataset in datasets:
            df_rank[dataset] = df_redperc.rank(axis=1)[dataset].value_counts()
        df_rank = df_rank.fillna(0)/neccs
        for dataset in datasets:
            for i in df_rank.index:
                results_plca[dataset][f'matred_rank_{int(i)}'] = df_rank.loc[i, dataset]
            results_plca[dataset]['matred_rank'] = list(df_redperc.mean().sort_values(ascending=True).index).index(dataset)+1
            results_plca[dataset]['matred_norm'] = (df_redperc.mean()/np.abs(df_redperc.mean().mean()))[dataset]

        ######################################################
        # Save results
        ######################################################
        results_meta[pewt][nplca] = results_plca

In [ ]:
dct_resultlabels = {
    'capecc_norm': f'EC Reduction (Cap ECC),\nNormalized',
    'capecc_perc_mean': f'EC Reduction (Cap ECC)\nMean Percentage',
    'capecc_perc_stdev': f'EC Reduction (Cap ECC)\nSt Dev Percentage',
    'capecc_rank': f'EC Reduction (Cap ECC)\nRank',
    'capecc_rank_1': f'EC Reduction (Cap ECC)\nRank #1 Frequency',
    'capecc_rank_2': f'EC Reduction (Cap ECC)\nRank #2 Frequency',
    'capecc_rank_3': f'EC Reduction (Cap ECC)\nRank #3 Frequency',
    'capecc_rank_4': f'EC Reduction (Cap ECC)\nRank #4 Frequency',
    'capecc_red_mean': f'EC Reduction (Cap ECC)\nMean',
    'capecc_red_stdev': f'EC Reduction (Cap ECC)\nSt Dev',
    'eci_cov': f'ECI COV',
    'eci_mean': f'ECI Mean',
    'eci_meannorm': f'ECI Mean\nNormalized',
    'eci_meanrank': f'ECI Mean\nRank',
    'eci_perc_mean': f'ECI Percentage\nMean',
    'eci_perc_norm': f'ECI Percentage\nNormalized',
    'eci_perc_rank': f'ECI Percentage\nRanked',
    'eci_perc_std': f'ECI Percentage\nSt Dev',
    'eci_rank_1': f'ECI Rank #1\nFrequency',
    'eci_rank_2': f'ECI Rank #2\nFrequency',
    'eci_rank_3': f'ECI Rank #3\nFrequency',
    'eci_rank_4': f'ECI Rank #4\nFrequency',
    'eci_std': f'ECI StDev',
    'eci_stdnorm': f'ECI StDev\nNormalized',
    'eci_stdrank': f'ECI StDev\nRank',
    'matred_norm': f'EC Reduction (Mat Red),\nNormalized',
    'matred_perc_mean': f'EC Reduction (Mat Red)\nMean Percentage',
    'matred_perc_stdev': f'EC Reduction (Mat Red)\nSt Dev Percentage',
    'matred_rank': f'EC Reduction (Mat Red)\nRank',
    'matred_rank_1': f'EC Reduction (Mat Red)\nRank #1 Frequency',
    'matred_rank_2': f'EC Reduction (Mat Red)\nRank #2 Frequency',
    'matred_rank_3': f'EC Reduction (Mat Red)\nRank #3 Frequency',
    'matred_rank_4': f'EC Reduction (Mat Red)\nRank #4 Frequenc2',
    'matred_red_mean': f'EC Reduction (Mat Red)\nMean',
    'matred_red_stdev': f'EC Reduction (Mat Red)\nSt Dev',
    'ui': 'Uncertainty Index',
    'ui_norm': f'Uncertainty Index\nNormalized',
    'ui_rank': f'Uncertainty Index\nRank',
}

## Raw results for meta analysis

In [ ]:
%%time
# extract all results into a dictionary with a dataframe for each PEWT
plca = 0
dataset = list(results_meta[pewt][plca].keys())[0]
columns = results_meta[pewt][plca][dataset].keys()
dct_allresults = {pewt:pd.DataFrame(columns=columns) for pewt in PEWT}

cols_results = [ele for ele in columns if ele not in ['plca', 'pewt', 'dataset']]
cols_results.sort()

# consolidate all results
for pewt in tqdm(PEWT, total=len(PEWT)):
    for plca in range(len(combos)):
        for dataset in results_meta[pewt][plca].keys():
            dct_allresults[pewt].loc[len(dct_allresults[pewt])] = results_meta[pewt][plca][dataset]
    dct_allresults[pewt] = dct_allresults[pewt].set_index('dataset')

# get standard deviations of each result across PEWTs
df_stds = pd.DataFrame(columns=cols_results)
for result in tqdm(cols_results, total=len(cols_results)):
    dft = pd.DataFrame(columns=PEWT)
    for pewt in PEWT:
        dft[pewt] = dct_allresults[pewt][result]
    df_stds[result] = dft.std(axis=1)

# df_stds = df_stds.set_index('dataset')

In [ ]:
# # look at resulting metrics across PEWTs
# ncols = 6
# nrows = int(np.ceil(len(cols_results)/ncols))

# fig, axes = plt.subplots(nrows, ncols, figsize=(10,nrows), gridspec_kw=dict(hspace=1.0))
# row, col = (0, 0)
# ax2del = list(product(range(nrows), range(ncols)))
# for result in tqdm(cols_results, total=len(cols_results)):
#     if result.endswith('rank'):
#         continue
#     elif result in []:
#         continue
#     ax = axes[row, col]
#     ax2del.remove((row, col))
#     for pewt in PEWT:
#         data = dct_allresults[pewt][result]
#         color = dct_colors[pewt]
#         with warnings.catch_warnings():
#             warnings.simplefilter('ignore')
#             sns.kdeplot(data, color=color, ax=ax, label=pewt)

#     ax.spines[['top', 'right', 'left']].set_visible(False)
#     ax.get_yaxis().set_visible(False)
#     ax.set_title(result, fontsize=8)
#     ax.tick_params(axis='both', which='major', labelsize=6)
#     ax.set_xlabel('')
    
#     if (row, col) == (0, ncols-1):
#         ax.legend(bbox_to_anchor=(1,1), loc='upper left')
    
#     if col == ncols-1:
#         row += 1
#         col = 0
#     else:
#         col += 1

# for (row, col) in ax2del:
#     fig.delaxes(axes[row, col])



## Check for correlations between standard deviations of results and input metrics

In [ ]:
# correlation analysis - didn't yield anything useful
df_metrics_plca = df_metrics.loc[df_stds.index]
if any(df_metrics_plca.index != df_stds.index):
    raise ValueError('Indices dont match between df_metrics and df_stds')
    
df_corr_meta = pd.DataFrame(columns=['metric', 'result', 'pearson', 'spearman', 'kendall'])
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for metric, result in tqdm(product(df_metrics_plca.columns, df_stds.columns), total=len(df_metrics_plca.columns)*len(df_stds.columns)):
        if metric in ['mean_uw']:
            continue
        x = df_metrics_plca[metric].values.astype(float)
        y = df_stds[result].values.astype(float)
        
        pearson, pval = pearsonr(x, y)
        spearman, pval = spearmanr(x, y)
        kendall, pval = kendalltau(x, y)    

        df_corr_meta.loc[len(df_corr_meta)] = [metric, result, pearson, spearman, kendall]

df_corr_meta['corr_meanabs'] = np.abs(df_corr_meta[['pearson', 'spearman', 'kendall']]).mean(axis=1)
df_corr_meta['corr_maxabs'] = np.abs(df_corr_meta[['pearson', 'spearman', 'kendall']]).max(axis=1)

## How much do results differ?

In [ ]:
ranks = np.arange(nmats)+1
def compare_results(result, ax, s=0.001, scale=500, fontsize=10):
    # collect data
    df_result = pd.DataFrame(index=combos.flatten(), columns=PEWT)
    for pewt in PEWT:
        for plca in range(len(combos)):
            for dataset in results_meta[pewt][plca].keys():
                df_result.loc[dataset, pewt] = results_meta[pewt][plca][dataset][result]

    # plot
    dct_se = {pewt:np.array([]) for pewt in PEWT}
    dft = pd.DataFrame(data=0, index=ranks, columns=ranks)
    for pewt1, pewt2 in permutations(PEWT, 2):
        if pewt1 == pewt2:
            continue
        if result.endswith('rank'):
            for x, y in zip(df_result[pewt1], df_result[pewt2]):
                dft.loc[x, y] += 1
                dft.loc[y, x] += 1
        else:
            ax.scatter(df_result[pewt1], df_result[pewt2], color='black', s=s)
        
        # calculate squared error
        se = ((df_result[pewt1]-df_result[pewt2])**2).values
        dct_se[pewt1] = np.concatenate([dct_se[pewt1], se])
        dct_se[pewt2] = np.concatenate([dct_se[pewt2], se])

        # set limits
        # if lim != None:
        #     ax.set_xlim(lim[0],lim[1])
        #     ax.set_ylim(lim[0],lim[1])
        lo = df_result.min().min()
        hi = df_result.max().max()
        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)

        # set ticks and labels
        xticks = ax.get_xticks()
        yticks = ax.get_yticks()
        if len(xticks) <= len(yticks):
            ticks = [np.round(ele,2) for ele in xticks]
        else:
            ticks = [np.round(ele,2) for ele in yticks]
        ax.set_xticks(ticks, ticks, fontsize=fontsize)
        ax.set_yticks(ticks, ticks, fontsize=fontsize)

        # ax.plot([lo, hi],[lo, hi], color='red', linewidth=1.0, linestyle='--', zorder=25)
        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)

    if result.endswith('rank'):
        for i, j in product(dft.index, dft.columns):
            ax.scatter(i, j, color='black', s=(dft/dft.max().max()).loc[i,j]*scale)
        ax.set_xlim(0., 5.)
        ax.set_ylim(0., 5.)
        ax.set_xticks(ranks, ranks)
        ax.set_yticks(ranks, ranks)

    ax.set_title(dct_resultlabels[result],fontsize=fontsize)
    
    # add NRMSE as single metric
    se = np.array(pd.DataFrame(dct_se)).flatten()
    nrmse = np.mean(se)**0.5/np.std(np.array(df_result).flatten())
    ax.text(0.02, 0.98, '{:.3f}'.format(np.round(nrmse,3)), fontsize=fontsize, ha='left', va='top', transform=ax.transAxes)
    
    return df_result, nrmse
        

result = 'capecc_norm'
fig, ax = plt.subplots(figsize=(4,4), dpi=75)
df_result, nrmse = compare_results(result, ax=ax)

In [ ]:
# look at the difference between some result categories
# dataset = list(results_meta[pewt][plca])[0]
# results_meta[pewt][plca][dataset]

results = list(df_stds.columns)

results_plot = [
    'eci_perc_mean',
    'ui',
    'matred_rank_1',
    'eci_meanrank',
    'ui_rank',
    'capecc_rank_1',
]

ncols = 3
nrows = int(np.ceil(len(results_plot)/ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6,nrows*1.75), gridspec_kw=dict(hspace=0.5, wspace=0.3))
row, col = (0,0)
ax2del = list(product(range(nrows), range(ncols)))
i = 0
for result in tqdm(results_plot, total=len(results_plot)):
    ax = axes[row, col]
    ax2del.remove((row, col))

    # plot
    df_result, nrmse = compare_results(result, ax=ax, s=0.00001, scale=200, fontsize=6)
    
    # format
    ax.text(-0.05,1.05,f'({alphabet[i]})', ha='right', va='bottom', transform=ax.transAxes, fontsize=8, weight='bold')
    i += 1
    ax.tick_params(axis='both', which='major', labelsize=6)
    ax.set_title(dct_resultlabels[result], fontsize=8)

    if (row, col) == (0,0):
        ax.text(0.2,0.98,'=NRMSE', ha='left', va='top', transform=ax.transAxes, fontsize=6)
    
    if col == ncols-1:
        row += 1
        col = 0
    else:
        col += 1

for (row, col) in ax2del:
    fig.delaxes(axes[row, col])


plt.savefig('../outputs/figures/CompareUQMethods_FIG_ScatterPlot_UQResults_Subset.png', bbox_inches='tight')

In [ ]:
# look at all the result categories
# dataset = list(results_meta[pewt][plca])[0]
# results_meta[pewt][plca][dataset]

results = list(df_stds.columns)
results_plot = results

# analyze data to figure out when the highest deviations occur
df_resultstd = pd.DataFrame(index=DATA.keys(), columns=results)
dct_nrmse = {}

ncols = 6
nrows = int(np.ceil(len(results_plot)/ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*1.5,nrows*1.5), gridspec_kw=dict(hspace=0.75, wspace=0.3))
row, col = (0,0)
ax2del = list(product(range(nrows), range(ncols)))
for result in tqdm(results_plot, total=len(results_plot)):
    ax = axes[row, col]
    ax2del.remove((row, col))
    df_result, nrmse = compare_results(result, ax=ax, s=0.00001, scale=100, fontsize=4)
    df_resultstd[result] = df_result.std(axis=1) / np.std(np.array(df_result).flatten())
    dct_nrmse[result] = nrmse
    
    ax.tick_params(axis='both', which='major', labelsize=4)
    ax.set_title(dct_resultlabels[result], fontsize=6)
    
    if (row, col) == (0,0):
        ax.text(0,0.98,'NRMSE=',ha='right', va='top', fontsize=4, transform=ax.transAxes)
    
    if col == ncols-1:
        row += 1
        col = 0
    else:
        col += 1

for (row, col) in ax2del:
    fig.delaxes(axes[row, col])

plt.savefig('../outputs/figures/CompareUQMethods_SUPP_ScatterPlot_UQResults_All.png', bbox_inches='tight')

In [ ]:
pd.Series(dct_nrmse).sort_values(ascending=False)

In [ ]:
# the results with the highest values had the highest standard deviations
np.abs(df_resultstd).mean().sort_values(ascending=False).iloc[:10]


In [ ]:
# the datasets with the highest values differed the most
df_resultstd.mean(axis=1).sort_values(ascending=False).iloc[:10]

In [ ]:
def compare_results_bypewt(result, ax, annot=True, fmt='.2f', annotsize=6, vmin=None, vmax=None, cbarbool=True):
    # prepare variables
    colors = sns.color_palette('Paired', len(PEWT))
    dct_color = {k:v for (k,v) in zip(PEWT, colors)}

    # collect data
    df_result = pd.DataFrame(index=combos.flatten(), columns=PEWT)
    for pewt in PEWT:
        for plca in range(len(combos)):
            for dataset in results_meta[pewt][plca].keys():
                df_result.loc[dataset, pewt] = results_meta[pewt][plca][dataset][result]

    # prepare df_plot
    df_plot = pd.DataFrame(index=PEWT, columns=PEWT)
    for pewt1, pewt2 in permutations(PEWT, 2):
        if pewt1 == pewt2:
            continue
        
        # calculate squared error
        nrmse = np.mean(((df_result[pewt1]-df_result[pewt2])**2).values)**0.5/np.std(np.array(df_result).flatten())
        df_plot.loc[pewt1, pewt2] = nrmse
        # df_plot.loc[pewt1, pewt1] = 0
    
    # plot
    # sns.heatmap(df_plot.iloc[:-1,1:].astype(float), cmap='Blues', annot=True, fmt='.3f')
    sns.heatmap(df_plot.astype(float), cmap='Blues', annot=annot, fmt=fmt, ax=ax, vmin=vmin, vmax=vmax, cbar=cbarbool, annot_kws=dict(size=annotsize))
    
    return df_plot

# result = 'ui'
# fig, ax = plt.subplots()
# df_plot = compare_results_bypewt(result, ax=ax)
# ax.set_title(result)


In [ ]:
df_result_bypewt = pd.DataFrame(columns=['result', 'pewt1', 'pewt2', 'wass', 'wassnorm'])

results = df_stds.columns

ncols = 6
nrows = int(np.ceil(len(results)/ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*1.5,nrows*1.5), gridspec_kw=dict(hspace=0.5, wspace=0.3))
row, col = (0,0)

ax2del = list(product(range(nrows), range(ncols)))
for result in tqdm(results, total=len(results)):
    ax = axes[row, col]
    ax2del.remove((row,col))
    df_plot = compare_results_bypewt(result, ax=ax, annot=True, annotsize=4, vmin=0.0, vmax=2.0, cbarbool=False)
    
    ax.tick_params(axis='both', which='major', labelsize=6)
    ax.set_title(dct_resultlabels[result], fontsize=6)
    
    if (row, col) != (0, 0):
        ax.set_xticks([])
        ax.set_yticks([])
    else:
        ax.set_xticks([])
        ax.set_yticks(np.arange(0,len(PEWT))+0.5, PEWT)
        
    # save results
    for pewt1, pewt2 in permutations(PEWT, 2):
        if np.mean(df_plot) != 0:
            df_plot_norm = df_plot/np.mean(df_plot) 
        else:
            df_plot_norm = df_plot.copy() #since eci_detrank are all the same
        wassnorm = df_plot_norm.loc[pewt1, pewt2]
        wass = df_plot.loc[pewt1, pewt2]
        dct = {
            'result': result,
            'pewt1': pewt1,
            'pewt2': pewt2,
            'wass': wass,
            'wassnorm': wassnorm,
        }
        df_result_bypewt.loc[len(df_result_bypewt)] = dct
    
    
    if col == ncols-1:
        row += 1
        col = 0
    else:
        col += 1

for (row, col) in ax2del:
    fig.delaxes(axes[row, col])

plt.savefig('../outputs/figures/CompareUQMethods_SUPP_AllResultsByAllUQMethods.png', bbox_inches='tight')


In [ ]:
# metric = 'wassnorm'
# df_heatmap = pd.DataFrame(index=PEWT, columns=PEWT)

# # fig, axes = plt.subplots(len(PEWT), len(PEWT), sharex=True, sharey=True)
# for (row, pewt1), (col, pewt2) in product(enumerate(PEWT), enumerate(PEWT)):
# #     if pewt1 == pewt2: fig.delaxes(axes[row, col])
# #     ax = axes[row, col]
#     dft = df_result_bypewt.loc[df_result_bypewt['pewt1'] == pewt1].loc[df_result_bypewt['pewt2'] == pewt2]
#     df_heatmap.loc[pewt1, pewt2] = dft[metric].mean()
# #     with warnings.catch_warnings():
# #         warnings.simplefilter('ignore')
# #         # sns.kdeplot(dft[metric], ax=ax)
# #         sns.stripplot(dft[metric], orient='h', ax=ax)
    
# #     # format
# #     ax.get_yaxis().set_visible(False)
# #     ax.set_ylabel('')
# #     ax.set_xlabel('')
# #     ax.set_xticklabels([])

# sns.heatmap(df_heatmap.astype(float), cmap='Blues', annot=True, fmt='.2f', cbar=False)

# plt.savefig('../outputs/figures/CompareUQMethods_FIG_MeanNRMSEBetweenPEWTs.png', bbox_inches='tight')

## How does wass distance between PDFs predict difference in results?

In [ ]:
# calculate wass distance between UQ fits and normalized error of results

results = list(dct_resultlabels.keys())
results.sort()

dct_resulterrors = {k:[] for k in results}
dct_resulterrors['dataset'] = []
dct_resulterrors['pewt1'] = []
dct_resulterrors['pewt2'] = []
dct_resulterrors['w1'] = []
dct_resulterrors['w2'] = []
dct_resulterrors['ks'] = []

# pull all results across all PEWTs and datasets
result_stds = {}
for result in results:
    dft = pd.DataFrame(columns=PEWT)
    for pewt in PEWT:
        dft[pewt] = dct_allresults[pewt][result]
    result_stds[result] = np.std(np.array(dft).flatten())

for dataset in tqdm(DATA, total=len(DATA)):
    for pewt1, pewt2 in combinations(PEWT,2):

        # calculate distance between plots
        data = DATA[dataset]['data']
        std = np.max([DATA[dataset]['metrics']['coeffvar'], DATA[dataset]['metrics']['coeffvar_uw']])
        lo = 0
        hi = np.max(data) + 10*std
        xplot = np.linspace(lo, hi, 1_000)
        func1 = prob_models[dataset][pewt1]
        func2 = prob_models[dataset][pewt2]
        ypdf1 = func1.pdf(xplot)
        ypdf2 = func2.pdf(xplot)
        ycdf1 = cumulative_trapezoid(ypdf1, xplot, initial=0)
        ycdf2 = cumulative_trapezoid(ypdf2, xplot, initial=0)
        ycdf1 = ycdf1 / np.max(ycdf1)
        ycdf2 = ycdf2 / np.max(ycdf2)

        # calculate metrics
        wass1 = wasserstein1_weighted(xplot, xplot, ypdf1, ypdf2, unitless=False)
        wass2 = wasserstein2_weighted(xplot, xplot, ypdf1, ypdf2, unitless=False)
        ksval = np.abs(ycdf1-ycdf2).max()

        # save metrics
        dct_resulterrors['dataset'].append(dataset)
        dct_resulterrors['pewt1'].append(pewt1)
        dct_resulterrors['pewt2'].append(pewt2)
        dct_resulterrors['w1'].append(wass1)
        dct_resulterrors['w2'].append(wass2)
        dct_resulterrors['ks'].append(ksval)

        # calculate difference in results
        for result in results:
            # calculate difference
            result1 = dct_allresults[pewt1].loc[dataset, result]
            result2 = dct_allresults[pewt2].loc[dataset, result]
            norm_error = np.abs(result1-result2) / result_stds[result]

            # save results
            dct_resulterrors[result].append(norm_error)

df_resulterrors = pd.DataFrame(dct_resulterrors)

In [ ]:
# see if KS test, W1, or W2 best predict different performance
# spearman, pearson, kendalltau
disttypes = ['ks', 'w1', 'w2']
corrtypes = ['spearman', 'pearson', 'kendalltau']

#initialize dictionary
dct_corr = {}
for dist in disttypes:
    dct_corr[dist] = {}
    for corr in corrtypes:
        dct_corr[dist][corr] = []

# calculate results
for result in results:
    for dist in disttypes:
        dct_corr[dist]['pearson'].append(np.abs(pearsonr(df_resulterrors[dist], df_resulterrors[result]).statistic))
        dct_corr[dist]['spearman'].append(np.abs(spearmanr(df_resulterrors[dist], df_resulterrors[result]).statistic))
        dct_corr[dist]['kendalltau'].append(np.abs(kendalltau(df_resulterrors[dist], df_resulterrors[result]).statistic))

In [ ]:
# plot distance calculation type vs correlation type
colors = sns.color_palette('Blues',3) + sns.color_palette('Greens',3) + sns.color_palette('Oranges',3)
i = 0

fig, ax = plt.subplots(figsize=(6,3))
for row, dist in enumerate(disttypes):
    for col, corr in enumerate(corrtypes):
        data = dct_corr[dist][corr]
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            sns.kdeplot(data, label=f'{dist}, {corr}', ax=ax, color=colors[i])
        i += 1

ax.legend(loc='upper left', bbox_to_anchor=(1,1))
ax.set_xlim(0., 1.)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.get_yaxis().set_visible(False)

In [ ]:
# plot wass distance vs difference between results
disttype = 'w1'
corrtype = 'pearson'

ncols = 6
nrows = int(np.ceil(len(results)/ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(10,nrows*1.5), sharex=True, sharey=True, gridspec_kw=dict(hspace=0.50))
row, col = (0,0)
ax2del = list(product(range(nrows), range(ncols)))
for result in results:
    ax = axes[row, col]
    ax2del.remove((row, col))
    xvals = df_resulterrors[disttype]
    yvals = df_resulterrors[result]

    if corrtype == 'pearson':
        rval = np.abs(pearsonr(xvals, yvals).statistic)
    elif corrtype == 'spearman':
        rval = np.abs(spearmanr(xvals, yvals).statistic)
    elif corrtype == 'kendalltau':
        rval = np.abs(kendalltau(xvals, yvals).statistic)
    else:
        raise ValueError(f'corrtype must be pearson, spearman, or kendalltau, instead of {corrtype}')
    ax.text(0.98,0.98,'{:.2f}'.format(np.round(rval,2)), ha='right', va='top', transform=ax.transAxes, fontsize=6)

    # plot
    ax.scatter(xvals, yvals, color='black', s=0.001)
    ax.set_title(dct_resultlabels[result], fontsize=6)
    ax.set_xlim(0,)
    ax.set_ylim(0,)
    ax.set_xticks([])
    ax.set_yticks([])

    if col == ncols-1:
        row += 1
        col = 0
    else:
        col += 1

for row, col in ax2del:
    fig.delaxes(axes[row, col])

plt.savefig('../outputs/figures/CompareUQMethods_WassVsResultDiff.png', bbox_inches='tight')

## Which metrics best predict results, and which results are best predicted?

In [ ]:
%%time
# calculate correlations between metrics and results
df_corr2 = pd.DataFrame(columns=['metric', 'result', 'pearson', 'spearman', 'kendall'])
metrics = list(dct_metriclabels.keys())
results = list(dct_resultlabels.keys())
metrics.sort()
results.sort()

for metric, result in tqdm(product(metrics, results), total=len(metrics)*len(results)):
    if metric == 'mean_uw': continue
    x = df_metrics_plca[metric].values.astype(float)
    y = df_resultstd[result].values.astype(float)
    
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        pearson, pval = pearsonr(x, y)
        spearman, pval = spearmanr(x, y)
        kendall, pval = kendalltau(x, y)

    df_corr2.loc[len(df_corr2)] = [metric, result, pearson, spearman, kendall]

df_corr2['corr_meanabs'] = np.abs(df_corr2[['pearson', 'spearman', 'kendall']]).mean(axis=1)
df_corr2['corr_maxabs'] = np.abs(df_corr2[['pearson', 'spearman', 'kendall']]).max(axis=1)
df_corr2 = df_corr2.dropna()


In [ ]:
# which correlations are the strongest? None of them. Super low correlations
df_corr2.sort_values('corr_maxabs', ascending=False).iloc[:10]

## Checking wass distance between UQ method and dataset vs. results

In [ ]:
# calculate KS-test, W-1, and W-2 between each probabilistic model and the weighted data
# ks_results = {pewt: [] for pewt in PEWT}
wass1_results = {pewt: [] for pewt in PEWT}
# wass2_results = {pewt: [] for pewt in PEWT}
for dataset in tqdm(DATA, total=len(DATA)):
    for pewt in PEWT:
        # get empirical data
        data = DATA[dataset]['data']
        weights = DATA[dataset]['weights']

        # create model PDF
        func = prob_models[dataset][pewt]
        std = np.max([np.std(data), weighted_std(data, weights)])
        lo = 0
        hi = np.max(data) + 10*std
        xplot = np.linspace(lo, hi, 1_000)
        ypdf = func.pdf(xplot)

        # # KS test
        # ycdf = cumulative_trapezoid(ypdf, xplot, initial=0)
        # ycdf = ycdf / np.max(ycdf)
        # x, y, funcecdf = weighted_ecdf(data, weights)
        # yecdf = funcecdf(xplot)
        # ksval = np.max(np.abs(ycdf-yecdf))
        # ks_results[pewt].append(ksval)

        # wass1
        wass1 = wasserstein1_weighted(data, xplot, weights, ypdf, unitless=False)
        wass1_results[pewt].append(wass1)

        # # wass2
        # wass2 = wasserstein2_weighted(data, xplot, weights, ypdf, unitless=False)
        # wass2_results[pewt].append(wass2)



# # KS
# df_ks = pd.DataFrame(ks_results, index=df_metrics.index)
# df_ks_rank = df_ks.rank(axis=1)
# df_ks_relative = df_ks.div(df_ks.mean(axis=1), axis=0)

# wass1
df_wass1 = pd.DataFrame(wass1_results, index=df_metrics.index)
df_wass1_rank = df_wass1.rank(axis=1)
df_wass1_relative = df_wass1.div(df_wass1.mean(axis=1), axis=0)

# # wass2
# df_wass2 = pd.DataFrame(wass2_results, index=df_metrics.index)
# df_wass2_rank = df_wass2.rank(axis=1)
# df_wass2_relative = df_wass2.div(df_wass2.mean(axis=1), axis=0)




In [ ]:
# plot all results as a functon of wass distance
results = list(dct_resultlabels.keys())
results.sort()
window = 501

ncols = 6
nrows = int(np.ceil(len(results)/ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(12,nrows*1.5), gridspec_kw=dict(hspace=0.75, wspace=0.6))
row, col = (0,0)
ax2del = list(product(range(nrows), range(ncols)))
for result in tqdm(results, total=len(results)):
    ax = axes[row, col]
    ax2del.remove((row, col))
    # plot
    xvals = []
    yvals = []
    for pewt in PEWT:
        dft = pd.concat([dct_allresults[pewt][result], df_wass1[pewt]], axis=1)
        trendline = dft.sort_values(pewt)[result].rolling(window=window, min_periods=1, center=True, closed='both').mean()
        ax.scatter(dft[pewt], dft[result], color=dct_colors[pewt], s=0.1/len(DATA))
        ax.plot(dft.sort_values(pewt)[pewt], trendline, color=dct_colors[pewt], linewidth=1.0, zorder=3, label=pewt)

        # add SRCCs
        xvals = np.concatenate([xvals, dft[pewt].values])
        yvals = np.concatenate([yvals, dft[result].values])
    srcc = spearmanr(xvals, yvals).statistic
    ax.text(1.0, 1.0, '{:.2f}'.format(srcc), color='black', ha='right', va='top', fontsize=6, transform=ax.transAxes)
    if (row, col) == (0,0):
        ax.text(1.0, 1.01, 'SRCC', color='black', ha='right', va='bottom', fontsize=6, transform=ax.transAxes)

    rect = matplotlib.patches.Rectangle((1.0-np.abs(srcc)*0.25, 0.85), np.abs(srcc)*0.25, 0.025, #left bottom width height
                                                facecolor='black', linewidth=1, transform=ax.transAxes)
    ax.add_patch(rect)


    # format
    ax.set_title(dct_resultlabels[result], fontsize=7)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_xlim(0,)
    ax.tick_params(axis='both', which='major', labelsize=4)

    if col == ncols-1:
        row += 1
        col = 0
    else:
        col += 1

for row, col in ax2del:
    fig.delaxes(axes[row, col])

ax = axes[0, ncols-1]
ax.legend(bbox_to_anchor=(1.05,0.75), loc='upper left', fontsize=7)

ax = axes[0,0]
ax.text(0,1.75, 'pLCA results (y-axis) vs Wasserstein-1 distance between UQ methods and datasets (x-axis) with rolling average trendlines', fontsize=10, ha='left', va='bottom', transform=ax.transAxes)

# Case study to showcase results

In [ ]:
# get the order of highest variation across a full pLCA for each combo
result = 'eci_perc_mean'
fig, ax = plt.subplots(figsize=(4,4), dpi=75)
df_result, nrmse = compare_results(result, ax=ax)

# variance across the whole plca, not just one dataset
var_by_combo = pd.Series(index=range(len(combos)), data=0.)
for nplca, combo in tqdm(enumerate(combos), total=len(combos)):
    for dataset in combo:
        var_by_combo[nplca] += df_stds.loc[dataset,result]

# get the datasets from the plca with the median variance cin this cateogry
median = int(len(var_by_combo)/2)
nplca = var_by_combo.sort_values().iloc[median-1:median].index[0] #median variance
nplcas = list(var_by_combo.sort_values().iloc[[0,median-1,-1]].index) #low, median, high



In [ ]:
# # find out which combo has the highest aggregate W1 score bewteeen UQ models
# dct_w1 = dict(df_wass1.mean(axis=1))
# dct_w1combo = {}

# for i, combo in enumerate(combos):
#     dct_w1combo[i] = 0.
#     for dataset in combo:
#         dct_w1combo[i] += dct_w1[dataset]

# srs = pd.Series(dct_w1combo).sort_values()

# median = int(len(srs)/2)
# nplca = srs.sort_values().iloc[median-1:median].index[0] #median variance
# nplcas = list(srs.sort_values().iloc[[0,median-1,-1]].index) #low, median, high

In [ ]:
# create heatmaps comparing UQ methods by wass score for selected pLCAs
# prepare dictionary to save wass scores

# get data and plot
vmin, vmax = (0., 0.25)
nrows = nmats
ncols = len(nplcas)
fig, axes = plt.subplots(nrows, ncols, figsize=(6,4), gridspec_kw=dict(hspace=0.5))
for col, nplca in enumerate(nplcas):
    for row, dataset in enumerate(combos[nplca]):
        ax = axes[row, col]

        # calculate wass
        df_heatmap = pd.DataFrame(index=PEWT, columns=PEWT)
        data = DATA[dataset]['data']
        weights = DATA[dataset]['weights']
        std = np.max([np.std(data), weighted_std(data, weights)])
        lo = np.max([0, np.min(data)-3*std])
        hi = np.max(data)+3*std
        xplot = np.linspace(lo, hi, 1_000)
        for pewt1, pewt2 in combinations(PEWT, 2):
            func1 = prob_models[dataset][pewt1]
            func2 = prob_models[dataset][pewt2]
            ypdf1 = func1.pdf(xplot)
            ypdf2 = func2.pdf(xplot)
            wass1 = wasserstein1_weighted(xplot, xplot, ypdf1, ypdf2, unitless=False)

            df_heatmap.loc[pewt1, pewt2] = wass1
            df_heatmap.loc[pewt2, pewt1] = wass1
        
        # plot
        sns.heatmap(df_heatmap.astype(float), cmap='Blues', annot=True, fmt='.2f', ax=ax, cbar=False, annot_kws={'size': 4}, vmin=vmin, vmax=vmax)

        # format
        ax.set_title(dataset, fontsize=6)
        if (row, col) != (0, 0):
            ax.set_xticks([])
            ax.set_yticks([])
        else:
            ax.set_xticks([])
            ax.set_yticks(np.arange(0,len(PEWT))+0.5, PEWT, fontsize=4, rotation=0)


ax = axes[0,0]
text = dct_resultlabels[result].replace('\n',' ')
ax.text(0,1.6, f"Wasserstein-1 distances between UQ fits for each dataset", fontsize=9, ha='left', va='bottom', transform=ax.transAxes)
ax.text(0,1.35, f'Lowest NRMSE', fontsize=7, ha='left', va='bottom', transform=ax.transAxes)

ax = axes[0,1]
ax.text(0,1.35, f'Median NRMSE', fontsize=7, ha='left', va='bottom', transform=ax.transAxes)

ax = axes[0,2]
ax.text(0,1.35, f'Highest NRMSE', fontsize=7, ha='left', va='bottom', transform=ax.transAxes)


plt.savefig('../outputs/figures/CompareUQMethods_FIG_PLCAW1Distances.png', bbox_inches='tight')



In [ ]:
# plot the components and total of each pLCA
nrows = nmats + 1
ncols = len(nplcas)
result = 'eci_perc_mean'

xplot = np.linspace(0,4,1_000)
fig, axes = plt.subplots(nrows, ncols, sharex=False, figsize=(10,nrows), gridspec_kw=dict(hspace=0.5, height_ratios=[1, 1, 1, 1, 2]))

# show data and fits
for col, nplca in enumerate(nplcas):
    datasets = combos[nplca]
    for row, dataset in enumerate(datasets):
        ax = axes[row, col]
        data = DATA[dataset]['data']
        weights = DATA[dataset]['weights']
        ax.scatter(data, [0]*len(data), s=weights/np.max(weights)*250, color='black', marker='|')

        for pewt in PEWT:
            func = prob_models[dataset][pewt]
            yplot = func.pdf(xplot)
            ax.plot(xplot, yplot, color=dct_colors[pewt], label=pewt)

        # insert rectangles for uncertainty index
        ygap = 0.05
        mag = 0.70
        xloc = 0.98
        yloc = 0.9
        # if (row, col) == (0,0):
        #     ax.text(xloc+0.01, yloc, 'Uncert.\nIndices', ha='left', va='top', transform=ax.transAxes, fontsize=6)
        # for pewt in PEWT:
        #     yloc -= ygap
        #     width = np.max([0,results_meta[pewt][nplca][dataset]['ui']])
        #     rect = matplotlib.patches.Rectangle((xloc-width*mag, yloc), width*mag, ygap*0.90, #left, bottom, width, height
        #                                         facecolor=dct_colors[pewt], linewidth=1, transform=ax.transAxes)
        #     # ax.text(xloc, yloc, '{:.2f}'.format(np.round(width,2)), ha='left', va='bottom', fontsize=4, transform=ax.transAxes)
        #     ax.add_patch(rect)

        # add mean wass
        color = 'tab:purple'
        # yloc -= 4*ygap
        meanwass = df_resulterrors.groupby('dataset')['w1'].mean()[dataset]
        if (row, col) == (0,0):
            ax.text(xloc+0.01, yloc+ygap*1.5, 'Mean\nInter UQ\nW1 Dist.', ha='left', va='top', transform=ax.transAxes, color=color, fontsize=6)
        rect = matplotlib.patches.Rectangle((xloc-meanwass*mag, yloc), meanwass*mag, ygap*1.5, #left bottom width height
                                                facecolor=color, linewidth=1, transform=ax.transAxes)
        ax.add_patch(rect)
        yloc -= ygap
        ax.text(xloc, yloc, '{:.3f}'.format(meanwass), ha='right', va='top', transform=ax.transAxes, fontsize=6, color=color)


        # format
        ax.get_yaxis().set_visible(False)
        ax.spines[['top', 'right', 'left']].set_visible(False)
        ax.set_xlim(np.min(xplot), np.max(xplot))
        if row != nrows-2:
            ax.set_xticklabels([])
        ax.set_ylim(0,)
        ax.text(1.0,1.0,dataset, ha='right', va='bottom', fontsize=8, transform=ax.transAxes)
        if (row, col) == (0,ncols-1):
            ax.legend(bbox_to_anchor=(1.025,1), loc='upper left')


# plot wbECI as the last row
row = nrows-1
nruns = 10_000
for col, nplca in enumerate(nplcas):
    ax = axes[row, col]
    datasets = combos[nplca]
    for pewt in PEWT:
        wbeci = np.zeros(nruns)
        for dataset in datasets:
            func = prob_models[dataset][pewt]
            if pewt.startswith('KDE'):
                wbeci += func.resample(nruns)[0]
            else:
                wbeci += func.rvs(nruns)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            sns.kdeplot(wbeci, color=dct_colors[pewt], ax=ax)
    ax.get_yaxis().set_visible(False)
    ax.set_xlim(0,np.max(xplot)*4)
    ax.spines[['top', 'right', 'left']].set_visible(False)

ax = axes[nrows-1, 0]
ax.text(1.0,1.0,'Sum across datasets', ha='right', va='top', fontsize=8, transform=ax.transAxes)


    
ax = axes[0,0]
text = dct_resultlabels[result].replace('\n',' ')
ax.text(0,1.75, f"pLCAs with the lowest, median and highest variance for {text}", fontsize=14, ha='left', va='bottom', transform=ax.transAxes)
ax.text(0,1.25, f'Lowest variance', fontsize=10, ha='left', va='bottom', transform=ax.transAxes)

ax = axes[0,1]
ax.text(0,1.25, f'Median variance', fontsize=10, ha='left', va='bottom', transform=ax.transAxes)

ax = axes[0,2]
ax.text(0,1.25, f'Highest variance', fontsize=10, ha='left', va='bottom', transform=ax.transAxes)


plt.savefig('../outputs/figures/CompareUQMethods_FIG_PLCAVisualizeUQFits.png', bbox_inches='tight')

In [ ]:
# percentage each dataset is at each rank
nruns = 10_000
nrows = 1
ncols = len(nplcas)
result = 'eci_perc_mean'
vmin = 0.2
vmax = 0.35


fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(8,2), gridspec_kw=dict(hspace=0.2))
for col, nplca in enumerate(nplcas):    
    # prepare data
    datasets = combos[nplca]
    dct_rank = pd.DataFrame(data=0., index=PEWT, columns=datasets)
    for pewt, dataset in product(PEWT, datasets):
        dct_rank.loc[pewt, dataset] = dct_allresults[pewt].loc[dataset,result]


    # plot
    rank = 1
    ax = axes[col]
    sns.heatmap(dct_rank, cmap='Blues', annot=True, fmt='.0%', cbar=False, ax=ax, vmin=vmin, vmax=vmax, annot_kws=dict(size=6))

    # format
    ax.text(-0.05,1.05,f'({alphabet[col]})', ha='right', va='bottom', transform=ax.transAxes, fontsize=8, weight='bold')
    ax.tick_params(axis='x', which='major', rotation=30)
    if col != 0:
        ax.set_yticklabels([])
    ymin, ymax = ax.get_ylim()
    xmin, xmax = ax.get_xlim()
    ax.hlines(0.0, xmin, xmax, linewidth=0.5, color='black', transform=ax.transAxes)
    ax.hlines(1.0, xmin, xmax, linewidth=0.5, color='black', transform=ax.transAxes)
    for xloc in np.arange(0,nmats+1)/nmats:
        ax.vlines(xloc, ymin, ymax, linewidth=0.5, color='black', transform=ax.transAxes)


# add titles
ax = axes[0]
ax.text(0,1.25, f'ECI percentage mean for each dataset and UQ method in three pLCAs', fontsize=12, ha='left', va='bottom', transform=ax.transAxes)
ax.text(0,1.05, f'Lowest variance', fontsize=10, ha='left', va='bottom', transform=ax.transAxes)

ax = axes[1]
ax.text(0,1.05, f'Median variance', fontsize=10, ha='left', va='bottom', transform=ax.transAxes)

ax = axes[2]
ax.text(0,1.05, f'Highest variance', fontsize=10, ha='left', va='bottom', transform=ax.transAxes)


plt.savefig('../outputs/figures/CompareUQMethods_FIG_RanksByDatasetAndPEWT.png', bbox_inches='tight')